# Continuing model experiments

The following set continues on from the previous round, focusing on filtered label catalogs (dropping false negatives sets from FTW) and combining Planet-only catalogs

In [1]:
import pandas as pd
from pathlib import Path
import yaml
import os
import re
import glob 
import geopandas as gpd
import shutil

# Catalogs

Catalog list:

- `ftw-catalog-fn-filtered.csv`: Dropped the FN labels, including out of bound Austria  labels. 
- `ftw-catalog-fns.csv`: List of FNs. We'll want to use these to drop names out of the Planet catalogs. 

In [2]:
catalog = pd.read_csv("../data/ftw-catalog.csv")
catalog_filtered = pd.read_csv("../data/ftw-catalog-fn-filtered.csv")

catalog_dropped = pd.read_csv("../data/ftw-catalog-fns.csv")
catalog = pd.read_csv("../data/ftw-catalog2.csv")
catalog_filtered2 = (catalog
                     .query("~name.isin(@catalog_dropped.name)")
                     .reset_index(drop=True))
catalog_filtered2.to_csv("../data/ftw-catalog-fn-filtered2.csv")

print(f"{len(catalog)} labels in original catalog")
print(f"{len(catalog_filtered)} labels in filtered full catalog")
print(f"{len(catalog)} labels in original catalog with presence-only dropped"\
      "from validation")
print(f"{len(catalog_filtered2)} labels in "\
      "filtered catalog that drops presence only from validation")


69360 labels in original catalog
69552 labels in filtered full catalog
69360 labels in original catalog with presence-only droppedfrom validation
68321 labels in filtered catalog that drops presence only from validation


## FTW Planet long + MA catalogs

Drop out FNs from it. 

In [3]:
ftwpl_pnir_catalog = pd.read_csv(
    "../data/ftw-planet-pseudo-nir-256-catalog.csv"
)
ftwpl_pnir_catalog["name2"] = (ftwpl_pnir_catalog["name"]
                               .str.rsplit("_", n=1).str[0])
ftwpl_pnir_catalog_filtrd = (ftwpl_pnir_catalog
                             .query("~name2.isin(@catalog_dropped.name)")
                             .reset_index(drop=True))

ftwpl_pnir_catalog_filtrd2 = ftwpl_pnir_catalog_filtrd.copy()
ftwpl_pnir_catalog_filtrd2["window_b"] = ftwpl_pnir_catalog_filtrd["window_a"]
ftwpl_pnir_catalog_filtrd_long = (
    pd.concat([ftwpl_pnir_catalog_filtrd, ftwpl_pnir_catalog_filtrd2], axis=0)
    .reset_index(drop=True)
)
ftwpl_pnir_catalog_filtrd_long["window_a"] = ""

# Long FTW catalog
ftwpl_pnir_catalog_filtrd_long.to_csv(
    "../data/ftw-planet-pseudo-nir-256-catalog-fn-filtered-long.csv", 
    index=False
)

# Long combined
ma_catalog = pd.read_csv("../data/mappingafrica-catalog.csv")
catalog_long = pd.concat([ftwpl_pnir_catalog_filtrd_long, ma_catalog], axis=0)

catalog_long.to_csv("../data/ftw-planet-pnir-mappingafrica-catalog.csv", 
                    index=False)

print(f"{len(ftwpl_pnir_catalog)} labels in original catalog")
print(f"{len(ftwpl_pnir_catalog_filtrd)} labels in filtered full catalog")
print(f"{len(ftwpl_pnir_catalog_filtrd_long)} labels in filtered Planet PNIR")
print(f"{len(catalog_long)} labels in combined catalog")


272080 labels in original catalog
267924 labels in filtered full catalog
535848 labels in filtered Planet PNIR
574924 labels in combined catalog


# Experiments

## Functions

In [4]:
def write_yaml(template_path: str, output_path: str, updates: dict = None):
    """
    Write a YAML file from a template file, with optional updates.

    Args:
        template_path (str): Path to the base YAML template file.
        output_path (str): Path to the output YAML file.
        updates (dict, optional): Dictionary of keys/values to update.
    """

    def recursive_update(d, u):
        for k, v in u.items():
            if isinstance(v, dict) and isinstance(d.get(k), dict):
                recursive_update(d[k], v)
            else:
                d[k] = v

    with open(template_path, 'r') as f:
        config = yaml.safe_load(f)
        if updates:
            recursive_update(config, updates)

    class IndentDumper(yaml.SafeDumper):
        def increase_indent(self, flow=False, indentless=False):
            return super().increase_indent(flow, False)

    # custom representer for lists
    def represent_list(dumper, data):
        # flow style only if all elements are scalars
        if all(isinstance(x, (str, int, float, bool, type(None))) for x in data):
            return dumper.represent_sequence("tag:yaml.org,2002:seq", data, 
                                             flow_style=True)
        else:
            return dumper.represent_sequence("tag:yaml.org,2002:seq", data, 
                                             flow_style=False)

    IndentDumper.add_representer(list, represent_list)

    with open(output_path, 'w') as f:
        yaml.dump(
            config,
            f,
            Dumper=IndentDumper,
            default_flow_style=False,  # keep dicts block-style
            sort_keys=False,
            indent=2,
            allow_unicode=True
        )


## Stats

In [6]:
band_stats = dict(
    t1 = dict(
        ma = {"min": [203.5], "max": [4590.4]}, 
        ma_s2 = {"min": [213.5], "max": [6079.5]},
        ma_s2sr = {"min": [211.5], "max": [6081]},
        ftw = {"min": [68.5], "max": [5772.3]},  # this should be same as t2
        full = {"min": [89.1], "max": [5528.3]},
        full_s2 = {"min": [87.7], "max": [5714]},
        full_s2sr = {"min": [88.2], "max": [5714.3]},
        planet_pnir = {"min": [56], "max": [4620]}
    ),
    t2 = dict(
        ma_s2pl = {"min": [169], "max": [5988]},
        ma_s2s2 = {"min": [158], "max": [6534]},
        ma_s2srs2sr = {"min": [181], "max": [6251]},
        ftw = {"min": [83], "max": [5637]},
        full_s2pl = {"min": [91], "max": [5708]},
        full_s2s2 = {"min": [89], "max": [5879]}, 
        full_s2srs2sr = {"min": [88], "max": [5758]},
        planet = {"min": [56], "max": [2214]}, 
        planet_pnir = {"min": [56], "max": [4620]}
    )
)

## FTW baseline with no false negatives

In [8]:
home_dir = "~/projects"
os.makedirs("../configs/additional", exist_ok=True)

In [30]:
cfg_name = "ftw2w-nofns-unet-effnb7-ltfl-sta-ep100"
update = dict()
update.setdefault("trainer", {})
update["trainer"] = dict(default_root_dir=f"~/working/models/{cfg_name}")
update.setdefault("model", {})
update["model"].setdefault("init_args", {})
update["model"]["init_args"].update(dict(in_channels=8))
update.setdefault("data", {})
update["data"].setdefault("init_args", {})
update["data"]["init_args"].update(
    dict(
        global_stats=band_stats['t2']['ftw'],
        random_shuffle=True,
        temporal_options="stacked",
        catalog=(
            f"{home_dir}/ftw-mappingafrica-integration/data/"\
            "ftw-catalog-fn-filtered2.csv"
        )   
    )
)
write_yaml("../configs/template-hpc-config.yaml", 
           f"../configs/additional/{cfg_name}.yaml", 
           updates=update)

## FTW planet pseudo-NIR 1 time point (long)

In [10]:
cfg_name = "ftw-planet-256pnir-unet-effnb7-ltfl-sta-ep100"
update = dict()
update.setdefault("trainer", {})
update["trainer"] = dict(default_root_dir=f"~/working/models/{cfg_name}")
update.setdefault("model", {})
update["model"].setdefault("init_args", {})
update["model"]["init_args"].update(dict(in_channels=8))
update.setdefault("data", {})
update["data"].setdefault("init_args", {})
update["data"]["init_args"].update(
    dict(
        crop_size=256,
        batch_size=64,
        global_stats=band_stats['t1']['planet_pnir'],
        catalog=(
            f"{home_dir}/ftw-mappingafrica-integration/data/"\
            "ftw-planet-pseudo-nir-256-catalog-fn-filtered-long.csv"
        )   
    )
)
write_yaml("../configs/template-hpc-config.yaml", 
           f"../configs/additional/{cfg_name}.yaml", 
           updates=update)